# AMADS Coding Notebooks

## Tonal Space

As discussed in the book
- **Tonnetze** ("tone networks") are graphs that arranges the musical objects
(typically single pitches or pitch combinations)
in a geometric shape.
- there are many ways of organising such tonal spaces.
- among the best known are the circle of fifths and the Euler tonnetz

This notebook explores
1. some visualisation / geometric layout of those space/s
2. the number of cycles through this space by chaining those PLR transforms

Other, related notebooks:
- For an empirical, corpus-based look at chord progressions rather than the abstract PLR-graph view here, see `chord_loops.ipynb`.
- For more on chord equivalence, see `chord_bigram.ipynb`.

---

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key:**
- <font color="green"> Green is for a block of information.
- <font color="purple"> Purple is for an exercise.
- <font color="crimson"> Crimson is for the solution to the exercise above it.

---

## <font color="green"> 1. PLR transforms

In the Euler tonnetz,
every major and minor triad is a *triangle*:
three pitch classes joined by
edges of a fifth and two kinds of third.
Crossing one of a triangle"s edges moves to a neighbouring triad
and the three possible moves correspond to the
**P**, **L**, and **R** transformations of neo-Riemannian theory:
- **P** (parallel): C major &#8596; c minor
- **L** (leading-tone exchange): C major &#8596; e minor
- **R** (relative): C major &#8596; a minor

As these are relations between triads they can also be set out on any other tonal space for those triads.
This notebook uses AMADS"s `EulerTonnetz` / `EulerTriad` implementation
to make that geometry concrete, using the circle of fifths design.
Then turns to **cycles** through this space.

First, in getting set up note that we'll use `networkx` for the grpahing.

This it not a core AMADS dependency; you may need to install (uncomment the line below)

In [ ]:
# %pip install networkx

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from amads.harmony.tonnetze.euler import EulerTriad

NOTE_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

TRANSFORM_ATTR = {
    "parallel": "p_transform",
    "leading_tone_exchange": "l_transform",
    "relative": "r_transform",
}
TRANSFORM_LETTER = {"parallel": "P", "leading_tone_exchange": "L", "relative": "R"}


def chord_pitches(root: int, major: bool):
    # a minimal representative chord (MIDI 60-71 range) for a given root/quality
    third = 4 if major else 3
    return [60 + root, 60 + (root + third) % 12, 60 + (root + 7) % 12]


def label(root: int, major: bool) -> str:
    return NOTE_NAMES[root] + ("" if major else "m")


def analyze(root: int, major: bool) -> EulerTriad:
    return EulerTriad(chord_pitches(root, major))


def apply_transform(root: int, major: bool, transform_name: str):
    """Apply one of P/L/R to the (root, major) triad, return the resulting (root, major)."""
    triad = analyze(root, major)
    getattr(triad, transform_name)()
    result_pitches = getattr(triad, TRANSFORM_ATTR[transform_name])
    result = EulerTriad(list(result_pitches))
    return (result.root, result.major_not_minor)


# sanity check against the class"s own docstring example
d_major = EulerTriad([2, 62, 6, 9])
d_major.leading_tone_exchange()
assert d_major.l_transform == (1, 61, 6, 9)

## <font color="purple"> Graphing the 24-triads

In the next few cells, we'll set up
a graph for the 24 triads (12 major and 12 minor)
and lines for the PLR transforms.

Explore the space and note the options for
- orderings (fiths or semitones)
- and alignment major triads on an outer ring with minor triads on an inner ring (P or R)

Q: Explore these options with the code provided, and consider which minimises line distances overall?

In [ ]:
G = nx.Graph()
for root in range(12):
    for major in (True, False):
        G.add_node((root, major))

for root in range(12):
    for major in (True, False):
        for transform_name in TRANSFORM_ATTR:
            neighbor = apply_transform(root, major, transform_name)
            G.add_edge((root, major), neighbor, transform=TRANSFORM_LETTER[transform_name])

print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
assert G.number_of_nodes() == 24
assert G.number_of_edges() == 36
assert all(d == 3 for _, d in G.degree())

In [ ]:
def tonnetz_layout(
        by_fifths: bool = False,
        inner_ring_p: bool = True,
        inner_ring_radius: float = 1.0,
        outer_ring_radius: float = 1.3
):
    """
    Design this notebook"s tonnetz graph style.

    Parameters
    ----------
    by_fifths:
        If False (default) then plot a circle of semi-tones.
        If True, then by fifths
    inner_ring_p:
        If True then align the inner-outer circles with the P relation (root aligned, e.g., C major and C minor).
        If False then use the R relation (e.g., C major and a minor).
        L is not implemented (the above two are the very dominant norms).
    """
    pos = {}
    for root in range(12):

        index = (root * 7) % 12 if by_fifths else root
        angle = np.pi / 2 - 2 * np.pi * index / 12  # C at the top, clockwise
        # TODO bonus experiment: swap the line above with that below for an alternative placing with C at the bottom, anticlockwise ;)
        # angle = 2 * np.pi * root / 12 - np.pi / 2

        # Major, outer
        pos[(root, True)] = (outer_ring_radius * np.cos(angle), outer_ring_radius * np.sin(angle))

        # Minor, inner
        minor_root = root if inner_ring_p else (root + 9) % 12
        pos[(minor_root, False)] = (inner_ring_radius * np.cos(angle), inner_ring_radius * np.sin(angle))
    return pos

POS = tonnetz_layout(by_fifths=True, inner_ring_p=True)  # TODO Adjust this for fifths vs semitones and ring alignment (p or r)
EDGE_COLOR = {"P": "#d62728", "L": "#1f77b4", "R": "#2ca02c"}


def draw_tonnetz(letters, title, ax=None, node_size=650):
    """Draw the 24-triad graph, showing only edges whose transform is in `letters`."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(6.5, 6.5))

    node_colors = ["#f4a259" if major else "#5b8bf7" for (_, major) in G.nodes()]
    nx.draw_networkx_nodes(G, POS, node_color=node_colors, node_size=node_size,
                            edgecolors="black", linewidths=0.6, ax=ax)
    nx.draw_networkx_labels(G, POS, labels={n: label(*n) for n in G.nodes()},
                             font_size=8, ax=ax)

    for letter in letters:
        edge_list = [(u, v) for u, v, d in G.edges(data=True) if d["transform"] == letter]
        nx.draw_networkx_edges(G, POS, edgelist=edge_list, edge_color=EDGE_COLOR[letter],
                                width=1.8, ax=ax)

    ax.set_title(title, fontsize=12)
    ax.set_aspect("equal")
    ax.axis("off")
    if standalone:
        plt.tight_layout()
        plt.show()


draw_tonnetz(["P", "L", "R"], "24 Triads and P- (red), L- (blue) and R- (green) transforms")


## <font color="crimson"> Solution

Of these

| Ring ordering | Inner relation | P step | L step | R step | Total |
|---------------|----------------|--------|--------|--------|-------|
| Fifths        | R (relative)   | −3     | 1      | 0      | 4     |
| Fifths        | P (parallel)   | 0      | 4      | 3      | 7     |
| Semitones     | R (relative)   | 3      | -5     | 0      | 8     |
| Semitones     | P (parallel)   | 0      | 4      | −3     | 7     |

So ordering by 5ths and using the R relation minimise these steps.
That may be part of the attraction to that ordering,
although this space is rarely used to show LPR relations, especially L,
and L is the transform that makes the difference.

---

## <font color="green"> 2. Cycles: PLR-based walks through the space

Set up:
here is a "walker" that ...
- ... starts at a triad,
- ... alternately apply two named transforms,
- ... keeps a note of the chord sequence traversed on the way
- ... stops when it reaches the triad it started at (this return making it a "cycle").


In [ ]:
def walk(start, transform_names, max_steps=200):
    """
    Alternate through `transform_names`,
    returning the sequence of (root, major) visited
    including the return to `start` (a cycle).
    """
    sequence = [start]
    current = start
    for i in range(max_steps):
        transform_name = transform_names[i % len(transform_names)]
        current = apply_transform(current[0], current[1], transform_name)
        sequence.append(current)
        if current == start:
            return sequence
    raise RuntimeError("Did not return to start within max_steps.")


def describe_walk(start, transform_names):
    seq = walk(start, transform_names)
    print("-".join(label(*t) for t in seq))
    print(f"({len(seq) - 1} steps back to start)")

## PL: 4x hexatonic cycles

Alternating P and L traces out a 6-cycle.

Repeating from different starting triads finds 4 distinct forms of this 6-step cycle,
together covering all 24 triads with no overlap.

These are the four "hexatonic systems" of neo-Riemannian theory
(Cohn calls them Northern, Southern, Eastern and Western "poles").

In [ ]:
describe_walk((0, True), ["parallel", "leading_tone_exchange"])

In [ ]:
pl_edges = [(u, v) for u, v, d in G.edges(data=True) if d["transform"] in ("P", "L")]
H_pl = nx.Graph(pl_edges)
pl_components = list(nx.connected_components(H_pl))
print(f"{len(pl_components)} components under P+L, sizes {sorted(len(c) for c in pl_components)}")
assert all(len(c) == 6 for c in pl_components)
assert all(d == 2 for _, d in H_pl.degree())  # confirms each component is a simple cycle, not just connected

draw_tonnetz(["P", "L"], "Alternating P/L: four disjoint hexatonic 6-cycles")


## RP: 3x octatonic cycles

Here we demo the same idea with alternating RP which pair produces:
- three distinct cycles
- each of 8-steps
- corresponding to the "octatonic system."


In [ ]:
describe_walk((0, True), ["relative", "parallel"])

In [ ]:
rp_edges = [(u, v) for u, v, d in G.edges(data=True) if d["transform"] in ("R", "P")]
H_rp = nx.Graph(rp_edges)
rp_components = list(nx.connected_components(H_rp))
print(f"{len(rp_components)} components under R+P, sizes {sorted(len(c) for c in rp_components)}")
assert all(len(c) == 8 for c in rp_components)
assert all(d == 2 for _, d in H_rp.degree())

draw_tonnetz(["R", "P"], "Alternating R/P: three disjoint octatonic 8-cycles")


## LR: a Hamiltonian cycle through all 24

This is the "special" case:
alternating L and R doesn't split into several smaller, distinct loops,
it produces a *single* cycle touching every major and minor triad exactly once.

This is a special kind of cycle called a Hamiltonian cycle.

In [ ]:
describe_walk((0, True), ["leading_tone_exchange", "relative"])

In [ ]:
lr_edges = [(u, v) for u, v, d in G.edges(data=True) if d["transform"] in ("L", "R")]
H_lr = nx.Graph(lr_edges)
lr_components = list(nx.connected_components(H_lr))
print(f"{len(lr_components)} component(s) under L+R, size {sorted(len(c) for c in lr_components)}")
assert len(lr_components) == 1
assert len(lr_components[0]) == 24
assert all(d == 2 for _, d in H_lr.degree())

draw_tonnetz(["L", "R"], "Alternating L/R: a single 24-triad Hamiltonian cycle")


## Is the LR alternation cycle special?

How many Hamiltonian Cycles are there through this space?

Is LR the only one?

If there are more, is there any structure or regularity among them?

In this section, we will address those questions by brute force,
exploring all options rather than getting into proofs.

This is tractable because every node has exactly one each of L, R, and P.
Any path-in removes one option, so the "choice" (branching factor)
is at most 2 at every step after the first move.

In [ ]:
def all_hamiltonian_cycles(graph):
    nodes = list(graph.nodes())
    adjacency = {v: list(graph.neighbors(v)) for v in nodes}
    start = nodes[0]
    found = []

    def backtrack(path, visited):
        if len(path) == len(nodes):
            if start in adjacency[path[-1]]:
                # each cycle can be walked in two directions from the fixed start;
                # keep only one by requiring the second node to precede the last.
                # See notes on "equivalence" below.
                if nodes.index(path[1]) < nodes.index(path[-1]):
                    found.append(list(path))
            return
        for candidate in adjacency[path[-1]]:
            if candidate not in visited:
                visited.add(candidate)
                path.append(candidate)
                backtrack(path, visited)
                path.pop()
                visited.discard(candidate)

    backtrack([start], {start})
    return found

In [ ]:
# We'll time this to show it's tractable
import time

t0 = time.time()
all_cycles = all_hamiltonian_cycles(G)
elapsed = time.time() - t0
print(f"{len(all_cycles)} Hamiltonian cycles found in {elapsed:.3f} seconds")

So there are 62 Cycles.

Is the LR alternation the only case of a repeating 2-transform pattern?

In [ ]:
def edge_transform_sequence(cycle):
    transforms = []
    for i in range(len(cycle)):
        u, v = cycle[i], cycle[(i + 1) % len(cycle)]
        transforms.append(G.edges[u, v]["transform"])
    return transforms


def is_pure_two_letter_alternation(transforms):
    if len(set(transforms)) != 2:
        return False
    return all(transforms[i] != transforms[i + 1] for i in range(len(transforms) - 1)) and transforms[0] != transforms[-1]


pure_alternations = [c for c in all_cycles if is_pure_two_letter_alternation(edge_transform_sequence(c))]
mixed = [c for c in all_cycles if c not in pure_alternations]

print(f"Pure two-letter alternating cycles: {len(pure_alternations)}")
for cycle in pure_alternations:
    print("  pattern:", "".join(edge_transform_sequence(cycle))[:8] + "...")

print(f"Everything else (uses 3 colours, or 2 colours non-alternating): {len(mixed)}")


Yes, it's the only 2-transform alternation.

## Is there structure among the 62? Well-trodden edges and symmetry

Are the 62 all unrelated?
Or are there:
- individual *edges* that get reused across many of them ("well-trodden edges"),
- or is there symmetry among the cycles as a group.

We take these in turn.

### Edge frequency

For each of the 36 edges, count how many of the 62 cycles pass through it.

In [ ]:
from collections import Counter

edge_usage = Counter()
for cycle in all_cycles:
    for i in range(len(cycle)):
        u, v = cycle[i], cycle[(i + 1) % len(cycle)]
        edge_usage[frozenset((u, v))] += 1

letter_of_edge = {frozenset((u, v)): d["transform"] for u, v, d in G.edges(data=True)}

by_letter = {"P": [], "L": [], "R": []}
for edge, count in edge_usage.items():
    by_letter[letter_of_edge[edge]].append(count)

for letter in "PLR":
    values = sorted(by_letter[letter])
    print(f"{letter}-edges: min {min(values)}, max {max(values)} across 62 cycles. So they are "
          f"{'all identical' if min(values) == max(values) else 'varying'} "
          f"({values[0]}/62 = {values[0]/62:.0%} of cycles use each one)"
          if min(values) == max(values) else f"{letter}-edges: usage varies, {values}")

No individual edge is "more travelled" than any other edge of the same transform.

That said, the *types* aren't equal:
every individual L-edge appears in more of the 62 cycles than every single P- or R-edge does.

Structurally, L-transforms are more central to how this graph can be toured than P or R.

### Equivalences

Among types of equivalences we can dispense with two readily:
rotation and retrograde equivalence are already built into how we defined "a cycle" above:
an unordered *set* of edges, not a directed sequence with a fixed start.

- **Rotation** (starting the walk from a different point on the same loop)
  doesn't create a different cycle, it's the same closed circuit.
- **Retrograde** (walking it backwards) doesn't either,
  every edge in this graph is undirected so
  the P/L/R transform between two triads doesn't depend on which direction you cross it.

Here"s a quick check of that expectation for the case of the LR cycle:
every rotation, in both directions, collapses back to the same edge set.

In [ ]:
lr_cycle = pure_alternations[0]
lr_edge_set = frozenset(frozenset((lr_cycle[i], lr_cycle[(i + 1) % len(lr_cycle)])) for i in range(len(lr_cycle)))

variants = set()
n = len(lr_cycle)
for start_idx in range(n):
    forward = lr_cycle[start_idx:] + lr_cycle[:start_idx]
    backward = list(reversed(forward))
    for variant in (forward, backward):
        es = frozenset(frozenset((variant[i], variant[(i + 1) % n])) for i in range(n))
        variants.add(es)

print(f"{2 * n} rotation/retrograde variants generated, all collapse to {len(variants)} distinct edge set(s)")


### Transposition

Shifting every triad"s root by the same number of semitones, $T_k$, is a
graph symmetry: it just relabels nodes, preserving which edges exist and
what colour they are. Does it also preserve the *set* of 62 Hamiltonian
cycles — and if so, how does it act on individual cycles?


In [ ]:
def transpose(node, k):
    root, major = node
    return ((root + k) % 12, major)


def apply_symmetry(edge_set_, fn, *args):
    return frozenset(frozenset(fn(v, *args) for v in edge) for edge in edge_set_)


all_edge_sets = [
    frozenset(frozenset((c[i], c[(i + 1) % len(c)])) for i in range(len(c)))
    for c in all_cycles
]
edge_set_lookup = {es: i for i, es in enumerate(all_edge_sets)}

# confirm T_k permutes the 62-cycle set for every k
assert all(
    set(apply_symmetry(es, transpose, k) for es in all_edge_sets) == set(all_edge_sets)
    for k in range(12)
)


def orbit_sizes(group_elements):
    # union-find the 62 cycles into orbits under the given list of node-mapping functions
    parent = list(range(len(all_cycles)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry

    for i, es in enumerate(all_edge_sets):
        for fn, args in group_elements:
            union(i, edge_set_lookup[apply_symmetry(es, fn, *args)])

    orbits = {}
    for i in range(len(all_cycles)):
        orbits.setdefault(find(i), []).append(i)
    return orbits


transposition_group = [(transpose, (k,)) for k in range(12)]
t_orbits = orbit_sizes(transposition_group)
print(f"{len(t_orbits)} orbits under transposition alone, sizes {sorted(len(v) for v in t_orbits.values())}")


So under transposition alone the 62 cycles fall into 9 orbits.

- Most cycles have a full orbit of 12 (shifting by any nonzero amount gives a different cycle),
- A few have partial symmetry
- Exactly **one** cycle is fixed by *every* transposition — an orbit of size 1. Guess which one?

It'll be useful to see the patterns here,
and we have something useful to that effect built for figuration:

In [ ]:
from amads.texture.figuration import chunk_by_pattern

for members in sorted(t_orbits.values(), key=lambda m: -len(m)):
    rep = all_cycles[members[0]]
    print(
        len(members),
        "".join(edge_transform_sequence(rep)),
        ["".join(x) for x in chunk_by_pattern(edge_transform_sequence(rep))]
    )

That size-1 orbit is the LR alternation cycle, but there are others with small periods of
- orbit 2 (sub pattern length 4)
- orbit 3 (sub pattern length 6)
- orbit 4 (sub pattern length 8)

The relations is given by
12 / orbit * pattern length = 24

### Inversion, and the T/I group

There"s one more equivalence/symmetry worth checking:
**pitch inversion** which reflects every pitch class through an axis $n$.

Applied to a triad this swaps major and minor
(inverting a major triad"s intervals produces a minor triad"s).

Inversion preserves the graph and moreover,
it preserves the P/L/R edges exactly
(a P-edge always maps to a P-edge, and so on).
Together, the 12 transpositions and 12 inversions form a 24-element group
(which music theory calls the "T/I group").

Here we redo the orbit analysis with this larger group.

In [ ]:
def inversion(node, axis):
    root, major = node
    return ((axis - root - 7) % 12, not major)

In [ ]:
inversion_group = [(inversion, (axis,)) for axis in range(12)]
ti_orbits = orbit_sizes(transposition_group + inversion_group)
print(f"{len(ti_orbits)} orbits under the full T/I group (order 24), "
      f"sizes {sorted(len(v) for v in ti_orbits.values())}")

for members in sorted(ti_orbits.values(), key=lambda m: -len(m)):
    rep = all_cycles[members[0]]
    print(
        len(members),
        "".join(edge_transform_sequence(rep)),
        ["".join(x) for x in chunk_by_pattern(edge_transform_sequence(rep))]
    )

Adding inversion merges two of the previous transposition-only orbits
(size 12 each)
into one orbit of size 24.

LR alternation is still the only Hamiltonian cycle on this graph
fixed by the entire 24-element T/I group.

So yes, it is special, at least mathematically.

Formally speaking, this is because the
T/I group and the
PLR group are
**dual** groups, meaning
(each is the centraliser of the other acting on the 24 triads).
See Lewin (1987) for more on this.

Both act transitively on the same 24 objects,
and the LR cycle sits at the intersection of that duality
- invariant under one whole group
- *generated by* elements of the other.

## Some References

- Euler, L. (1739). *Tentamen novae theoriae musicae...* Saint Petersburg Academy.
    - Seemingly the first to set out "the" Tonnetz.
- Lewin, D. (1987). *Generalized Musical Intervals and Transformations.* Yale University Press.
    - Source of the T/I and PLR group duality.


Ends

---